In [1]:
import pandas as pd
import numpy as np
import networkx as nx
import gc  # Garbage Collector for memory management

print("Environment Ready.")

Environment Ready.


In [2]:
# phase 1

# Limit rows initially to test your complete pipeline logic safely
SAMPLE_ROWS = 100000 

trans_cols = ['TransactionID', 'TransactionDT', 'TransactionAmt', 'card1', 'isFraud']
id_cols = ['TransactionID', 'DeviceInfo']

df_trans = pd.read_csv('C:/Users/20aya/my_project/data/train_transaction.csv', usecols=trans_cols, nrows=SAMPLE_ROWS)
df_id = pd.read_csv('C:/Users/20aya/my_project/data/train_identity.csv', usecols=id_cols, nrows=SAMPLE_ROWS)

# Merge and isolate rows containing physical device connections
df_clean = pd.merge(df_trans, df_id, on='TransactionID', how='left').dropna(subset=['card1', 'DeviceInfo']).copy()

# Explicitly clean up individual dataframes to free RAM
del df_trans, df_id
gc.collect() 

print(f"Data ready for graph injection. Shape: {df_clean.shape}")

Data ready for graph injection. Shape: (36465, 6)


In [3]:
import pandas as pd
import numpy as np

print("Injecting Temporal and Velocity Features...")

# Ensure the data is strictly chronological
df_clean = df_clean.sort_values('TransactionDT')

# Feature 1: Cumulative Transaction Count (Velocity)
# How many times has this card been used BEFORE this specific transaction?
# A value of 0 means it's a brand new card (high risk if the amount is also high).
df_clean['card_cumulative_txns'] = df_clean.groupby('card1').cumcount()

# Feature 2: Time Since Last Transaction (Temporal Lag)
# 1. Group by card and shift the timestamp down by 1 to get the PREVIOUS transaction time
df_clean['previous_txn_time'] = df_clean.groupby('card1')['TransactionDT'].shift(1)

# 2. Calculate the difference in seconds
df_clean['time_since_last_txn'] = df_clean['TransactionDT'] - df_clean['previous_txn_time']

# 3. Fill NaNs (which represent the very first time a card is seen) with a massive 
# number (-999 or a distinct outlier) so XGBoost knows it's a first-time event.
df_clean['time_since_last_txn'] = df_clean['time_since_last_txn'].fillna(-999)

# Clean up the temporary column
df_clean = df_clean.drop(columns=['previous_txn_time'])

print("Preview of new features for a highly active card:")
#print(df_clean.groupby('card1')['TransactionAmt'].sum().head(10))



# Let's peek at a card that has been used multiple times to verify the logic
active_card = df_clean[df_clean['card_cumulative_txns'] > 2]['card1'].iloc[0]
#print(df_clean[df_clean['card1'] == active_card][['card1','TransactionDT', 'TransactionAmt', 'card_cumulative_txns', 'time_since_last_txn']])


Injecting Temporal and Velocity Features...
Preview of new features for a highly active card:


In [4]:
#phase 2
import networkx as nx
import pandas as pd

# What we are doing: Prefixing IDs to guarantee they are unique entity types.
# Why we are doing it: To force the graph into a 'Bipartite' structure (Cards strictly connect to Devices).

print("Preparing Node Identifiers...")
df_clean['card_node'] = "CARD_" + df_clean['card1'].astype(str)
df_clean['device_node'] = "DEV_" + df_clean['DeviceInfo'].astype(str)

# Initialize an empty, undirected graph
G = nx.Graph()
print("Empty NetworkX Graph initialized.")



print("Building the spatial network... (This may take a moment)")




Preparing Node Identifiers...
Empty NetworkX Graph initialized.
Building the spatial network... (This may take a moment)


In [5]:
# What we are doing: Rapidly looping through the transaction records to draw network edges.
# Why we are doing it: To establish the physical topology. If an edge already exists (the same card used on the same device twice), we increase its weight rather than overwriting it.

edges_data = zip(
    df_clean['card_node'], 
    df_clean['device_node'], 
    df_clean['TransactionAmt'], 
    df_clean['isFraud']
)

for card, device, amt, is_fraud in edges_data:
    # If the connection already exists, update the metrics on that connection
    if G.has_edge(card, device):
        G[card][device]['transaction_count'] += 1
        G[card][device]['cumulative_amt'] += amt
        if is_fraud == 1:
            G[card][device]['fraud_count'] += 1
    else:
        # If it's a new connection, create the edge
        G.add_edge(
            card, 
            device, 
            transaction_count=1, 
            cumulative_amt=amt, 
            fraud_count=int(is_fraud)
        )

print(f"Graph Built Successfully!")
print(f"Total Unique Entities (Nodes): {G.number_of_nodes()}")
print(f"Total Connections (Edges): {G.number_of_edges()}")

Graph Built Successfully!
Total Unique Entities (Nodes): 5564
Total Connections (Edges): 10870


In [6]:
#phase 2 continue...
# What we are doing: Counting exactly how many distinct edges connect to every single node.
# Why we are doing it: Degree centrality measures velocity. A normal user uses 1 card on 1-2 devices (Degree = 1 or 2). A fraudster running a credential stuffing attack might test 50 cards on a single virtual machine (Device Degree = 50). This makes anomalies mathematically obvious.

print("Calculating Degree Centrality...")

# G.degree() returns a dictionary of {node_id: number_of_connections}
node_degrees = dict(G.degree())

# Map these calculated degrees back to the original tabular rows
df_clean['card_degree'] = df_clean['card_node'].map(node_degrees).fillna(0).astype(int)
df_clean['device_degree'] = df_clean['device_node'].map(node_degrees).fillna(0).astype(int)

print("Degree Centrality mapped to dataframe.")







Calculating Degree Centrality...
Degree Centrality mapped to dataframe.


In [7]:
# What we are doing: Identifying completely isolated sub-networks within the massive global graph.
# Why we are doing it: Fraud rings do not act alone; they rotate shared resources. If 4 stolen cards are being cycled through 3 specific IP/Device footprints, they form an isolated "island" (a connected component) of size 7. Normal users usually exist in isolated components of size 2 (1 card, 1 phone).

print("Applying Optimized Pruning Threshold...")

# Lock in the discovered optimal threshold
OPTIMAL_THRESHOLD = 7

# Find offenders
final_super_nodes = [node for node, degree in G.degree() if degree > OPTIMAL_THRESHOLD]

# Prune and calculate final components
G_final = G.copy()
G_final.remove_nodes_from(final_super_nodes)
final_components = list(nx.connected_components(G_final))

# Map cluster sizes
final_node_to_cluster = {}
for component in final_components:
    cluster_size = len(component)
    for node in component:
        final_node_to_cluster[node] = cluster_size

# Update dataframe
df_clean['network_cluster_size'] = df_clean['card_node'].map(final_node_to_cluster).fillna(1).astype(int)

# Drop string nodes to save memory, keeping only our engineered features and tabular data
df_final = df_clean.drop(columns=['card_node', 'device_node'])

# Save the finalized, mathematically enriched dataset
output_path = 'C:/Users/20aya/my_project/data/engineered_graph_features.csv'
df_final.to_csv(output_path, index=False)

print(f"Phase 2 Complete! File saved to {output_path}.")
print(df_final[['TransactionID', 'isFraud', 'card_degree', 'device_degree', 'network_cluster_size']].head())

Applying Optimized Pruning Threshold...
Phase 2 Complete! File saved to C:/Users/20aya/my_project/data/engineered_graph_features.csv.
    TransactionID  isFraud  card_degree  device_degree  network_cluster_size
4         2987004        0            2              4                     4
8         2987008        0           28           1646                     1
10        2987010        0            1           2843                     1
16        2987016        0            2           1259                     1
17        2987017        0            2           2843                     1
